In [1]:
#!pip install fastapi uvicorn pymongo yfinance nest_asyncio

In [2]:
import os
from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi
uri = os.environ["MONGODB_URI"]

client = MongoClient(uri, server_api=ServerApi('1'))

try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [3]:
from fastapi import FastAPI, Query
from typing import Optional, List
import nest_asyncio
import uvicorn
from pymongo import MongoClient
from datetime import datetime
import pandas as pd
import threading

In [4]:
from fastapi import FastAPI, Query, HTTPException
from pymongo import MongoClient
from datetime import datetime

app = FastAPI(title="SSE Historical Data API")

db = client["sse"]
collection = db["prices"]

In [5]:
from fastapi import FastAPI, HTTPException
from datetime import datetime
import nest_asyncio

app = FastAPI(title="SSE Historical Data API")

# -------------------------
# Helper: convert date field
# -------------------------
def parse_date_field(doc):
    """Convert date field to ISO string for JSON"""
    d = doc.get("date")
    if hasattr(d, "isoformat"):
        doc["date"] = d.isoformat()
    return doc

# -------------------------
# /prices endpoint
# -------------------------
@app.get("/prices")
def prices(
    ticker: str,
    start_date: str,
    end_date: str,
    fields: Optional[str] = Query(None, description="Comma-separated fields like high,low,volume")
):
    # 1️⃣ Parse dates
    try:
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
    except Exception:
        return {"error": "Invalid start_date or end_date format. Use YYYY-MM-DD."}

    # 2️⃣ Query MongoDB
    query = {
        "ticker": ticker,
        "date": {"$gte": start_dt, "$lte": end_dt}
    }
    cursor = collection.find(query)

    # 3️⃣ Convert to DataFrame
    df = pd.DataFrame(list(cursor))
    if df.empty:
        return {"error": f"No data found for {ticker} in the given date range."}

    # 4️⃣ Drop MongoDB _id field if exists
    if "_id" in df.columns:
        df.drop(columns=["_id"], inplace=True)

    # 5️⃣ Filter requested fields
    if fields:
        requested = [f.strip() for f in fields.split(",")]
        # Always include 'date' and 'ticker'
        requested = [c for c in requested if c in df.columns]
        requested = ["date", "ticker"] + requested
        df = df[requested]

    # 6️⃣ Convert dates to string for JSON
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

    return df.to_dict(orient="records")


In [6]:
import nest_asyncio
import uvicorn
import threading

nest_asyncio.apply()  # Allow async in notebook

def run_api():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Run in a separate thread so notebook stays interactive
thread = threading.Thread(target=run_api, daemon=True)
thread.start()


INFO:     Started server process [19260]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55594 - "GET /prices?ticker=600000.SS&start_date=2010-01-04&end_date=2010-01-05&fields=high%2Clow%2Cvolume HTTP/1.1" 200 OK
INFO:     127.0.0.1:55615 - "GET /prices?ticker=600000.SS&start_date=2010-01-01&end_date=2010-01-05&fields=high HTTP/1.1" 200 OK
INFO:     127.0.0.1:55627 - "GET /prices?ticker=600000.SS&start_date=2010-01-01&end_date=2010-01-09&fields=low HTTP/1.1" 200 OK
INFO:     127.0.0.1:55630 - "GET /prices?ticker=600000.SS&start_date=2010-01-01&end_date=2010-01-09&fields=low HTTP/1.1" 200 OK
